In [ ]:
# ==========================================
# 1. TEMİZ VE GÜNCEL KURULUM
# ==========================================
import os

print("⚙️ Eski dosyalar temizleniyor ve Google Chrome indiriliyor...")

# Önceki kurulumları temizle
os.system("rm -rf /usr/bin/google-chrome")
os.system("rm -rf /usr/bin/chromedriver")

# Gerekli kütüphaneleri kur
os.system("pip install selenium supabase beautifulsoup4 webdriver-manager -qq")

# 1. Google Chrome'un Orijinal .deb dosyasını indir ve kur
# (Chromium değil, Google Chrome Stable kullanıyoruz)
os.system("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb")
os.system("apt-get update -qq")
os.system("apt-get install -y ./google-chrome-stable_current_amd64.deb -qq")

print("✅ Kurulum tamamlandı.")

⚙️ Eski dosyalar temizleniyor ve Google Chrome indiriliyor...
✅ Kurulum tamamlandı.


In [ ]:
# ==========================================
# 2. DRIVER AYARLARI (OTO-GÜNCELLEMELİ)
# ==========================================
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

# Ayarlar
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--remote-debugging-port=9222') # Çökme önleyici port

# Driver'ı otomatik indir ve ayarla (En kritik kısım burası)
# ChromeDriverManager().install() komutu yüklü Chrome'a uygun driver'ı bulur.
service = Service(ChromeDriverManager().install())

# Test için tarayıcıyı başlatıp kapatıyoruz
try:
    driver_test = webdriver.Chrome(service=service, options=chrome_options)
    print("✅ Tarayıcı başarıyla başlatıldı! Sorun çözüldü.")
    driver_test.quit()
except Exception as e:
    print(f"❌ Hala hata var: {e}")

✅ Tarayıcı başarıyla başlatıldı! Sorun çözüldü.


In [ ]:
# ==========================================
# 3. SUPABASE BAĞLANTISI VE KEYLER
# ==========================================
from supabase import create_client, Client

# Buraya YENİ oluşturduğun keyleri gir
# Tırnakların içini doldur
MY_SUPABASE_URL = "https://qcytfdttggdutynnhfzx.supabase.co"
MY_SUPABASE_KEY = "sb_secret_3SiJIfxyRT6P-sBzD_7nuw_MBASeDIt"

try:
    # İstemciyi oluşturuyoruz
    supabase: Client = create_client(MY_SUPABASE_URL, MY_SUPABASE_KEY)
    print("✅ Supabase bağlantısı hazır.")
except Exception as e:
    print(f"❌ Bağlantı Hatası: {e}")

✅ Supabase bağlantısı hazır.


In [ ]:
# ==========================================
# 3. SCRAPING MOTORU (İY SKORU DÜZELTİLDİ - V8)
# ==========================================
import re
import ast
import time
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
import copy

# Yardımcı Fonksiyonlar
def clean_odd(value):
    if not value or value == '-' or value == '': return None
    try: return float(value.replace(',', '.'))
    except: return None

def parse_date(date_str):
    try:
        clean_str = date_str.replace('Tarih :', '').strip()
        dt_obj = datetime.strptime(clean_str, '%d.%m.%Y %H:%M')
        return dt_obj.strftime('%Y-%m-%d %H:%M:%S')
    except: return None

# Ana Fonksiyon
def process_full_match(match_url):
    chrome_options.page_load_strategy = 'eager'
    driver = webdriver.Chrome(service=service, options=chrome_options)

    try:
        print(f"🌍 Kontrol ediliyor: {match_url}")
        driver.get(match_url)
        time.sleep(2)

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # --- 0. OYNANMIŞ MI VE SKORLAR ---
        match_status = "Bilinmiyor"
        score_ft = None
        score_ht = None

        try:
            # Durum (MS)
            status_div = soup.find("div", id="dvStatusText") or soup.find("div", class_="match-time")
            if status_div: match_status = status_div.get_text(strip=True)

            # Maç Skoru (MS)
            ft_div = soup.find("div", id="dvScoreText") or soup.find("div", class_="match-score")
            if ft_div: score_ft = ft_div.get_text(strip=True)

            # İlk Yarı Skoru (İY) - DÜZELTİLEN KISIM
            ht_div = soup.find("div", id="dvHTScoreText") or soup.find("div", class_="hf-match-score")
            if ht_div:
                raw_ht = ht_div.get_text(strip=True) # Örn: "İY : 0 - 3" veya "(0-3)"

                # Hem "İY", hem ":", hem de parantezleri temizle
                score_ht = raw_ht.replace('İY', '').replace(':', '').replace('(', '').replace(')', '').strip()
        except: pass

        if match_status != "MS":
            print(f"⚠️ Maç Bitmemiş ({match_status}). Atlanıyor...")
            return

        # --- 1. MAÇ KODU ---
        try:
            match_code = match_url.lower().split('/mac/')[1].split('/')[0]
        except:
            match_code = str(int(time.time()))

        # --- 2. MAÇ DETAYLARI ---
        match_info = {"league": None, "season": None, "match_date": None}
        info_wrapper = soup.find("div", class_="match-info-wrapper-top")

        if info_wrapper:
            season_div = info_wrapper.find("div", class_="match-info-wrapper-season")
            if season_div:
                full_text = season_div.get_text(strip=True)
                match = re.search(r'(\d{4}/\d{4})', full_text)
                if match:
                    match_info["season"] = match.group(1)
                    match_info["league"] = full_text.replace(match.group(1), "").strip()
                else:
                    match_info["league"] = full_text

            date_div = info_wrapper.find("div", class_="match-info-date")
            if date_div:
                match_info["match_date"] = parse_date(date_div.get_text(strip=True))

        # --- 3. TAKIMLAR ---
        home, away = "Bilinmiyor", "Bilinmiyor"
        try:
            h_tag = soup.find("a", class_="left-block-team-name")
            a_tag = soup.find("a", class_="r-left-block-team-name")
            if h_tag: home = h_tag.get_text(strip=True)
            if a_tag: away = a_tag.get_text(strip=True)
        except: pass

        print(f"📌 {home} vs {away} | MS: {score_ft} | İY: {score_ht}")

        row = {
            "match_code": match_code,
            "home_team": home, "away_team": away,
            "league": match_info["league"], "season": match_info["season"],
            "match_date": match_info["match_date"],
            "score_ft": score_ft,
            "score_ht": score_ht,
            "status": match_status
        }

        # --- 4. ORANLAR ---
        bet_boxes = soup.find_all("div", class_="md")

        for box in bet_boxes:
            title_div = box.find("div", class_="detail-title")
            if not title_div: continue

            title_copy = copy.copy(title_div)
            if title_copy.find("span"):
                title_copy.find("span").decompose()

            market_name = title_copy.get_text(strip=True).replace(',', '.')

            link = box.find("a", href=True)
            if link and "openOddsDialog" in link['href']:
                match = re.search(r"openOddsDialog\((.*?)\)", link['href'])
                if match:
                    params = match.group(1)
                    arrays = re.findall(r"\[.*?\]", params)
                    if len(arrays) >= 2:
                        outcomes = ast.literal_eval(arrays[0])
                        raw_odds = ast.literal_eval(arrays[1])
                        odds_map = dict(zip(outcomes, raw_odds))

                        if market_name == "Maç Sonucu":
                            row["ms_1"] = clean_odd(odds_map.get('1'))
                            row["ms_x"] = clean_odd(odds_map.get('X'))
                            row["ms_2"] = clean_odd(odds_map.get('2'))
                        elif market_name == "Çifte Şans":
                            row["cs_1x"] = clean_odd(odds_map.get('1-X'))
                            row["cs_12"] = clean_odd(odds_map.get('1-2'))
                            row["cs_x2"] = clean_odd(odds_map.get('X-2'))
                        elif market_name == "İlk Yarı/Maç Sonucu":
                            row["iyms_11"] = clean_odd(odds_map.get('1/1'))
                            row["iyms_1x"] = clean_odd(odds_map.get('1/X'))
                            row["iyms_12"] = clean_odd(odds_map.get('1/2'))
                            row["iyms_x1"] = clean_odd(odds_map.get('X/1'))
                            row["iyms_xx"] = clean_odd(odds_map.get('X/X'))
                            row["iyms_x2"] = clean_odd(odds_map.get('X/2'))
                            row["iyms_21"] = clean_odd(odds_map.get('2/1'))
                            row["iyms_2x"] = clean_odd(odds_map.get('2/X'))
                            row["iyms_22"] = clean_odd(odds_map.get('2/2'))
                        elif "1.5 Alt/Üst" in market_name:
                            if "1. Yarı" not in market_name and "Evsahibi" not in market_name and "Deplasman" not in market_name:
                                row["au_15_alt"] = clean_odd(odds_map.get('Alt'))
                                row["au_15_ust"] = clean_odd(odds_map.get('Üst'))
                        elif "2.5 Alt/Üst" in market_name:
                            if "1. Yarı" not in market_name and "Evsahibi" not in market_name and "Deplasman" not in market_name:
                                row["au_25_alt"] = clean_odd(odds_map.get('Alt'))
                                row["au_25_ust"] = clean_odd(odds_map.get('Üst'))
                        elif market_name == "Karşılıklı Gol":
                            row["kg_var"] = clean_odd(odds_map.get('Var'))
                            row["kg_yok"] = clean_odd(odds_map.get('Yok'))
                        elif market_name == "Toplam Gol Aralığı":
                            row["tg_0_1"] = clean_odd(odds_map.get('0-1 Gol'))
                            row["tg_2_3"] = clean_odd(odds_map.get('2-3 Gol'))
                            row["tg_4_5"] = clean_odd(odds_map.get('4-5 Gol'))
                            row["tg_6_plus"] = clean_odd(odds_map.get('6+ Gol'))

        data = supabase.table("matches").upsert(row).execute()
        print(f"✅ KAYDEDİLDİ: {home} vs {away} (MS: {score_ft} / İY: {score_ht})")

    except Exception as e:
        print(f"❌ Hata: {e}")
    finally:
        driver.quit()

# TEST
process_full_match("https://arsiv.mackolik.com/Mac/4308325/Gaziantep-FK-Galatasaray")

🌍 Kontrol ediliyor: https://arsiv.mackolik.com/Mac/4308325/Gaziantep-FK-Galatasaray
📌 Gaziantep FK vs Galatasaray | MS: 0 - 3 | İY: 0 - 3
✅ KAYDEDİLDİ: Gaziantep FK vs Galatasaray (MS: 0 - 3 / İY: 0 - 3)


In [ ]:
# ==========================================
# 4. LİNK TOPLAYICI (LINK HARVESTER)
# ==========================================
import requests
from bs4 import BeautifulSoup
import time

def get_season_links(league_id, season_id, total_weeks=38):
    """
    Belirtilen lig ve sezonun tüm haftalarını gezerek maç linklerini toplar.
    Selenium kullanmaz, requests ile direkt sunucudan çeker (Çok Hızlı).
    """
    base_url = "https://arsiv.mackolik.com/AjaxHandlers/StandingHandler.aspx"
    all_match_links = []

    print(f"🔄 Link toplama işlemi başlıyor... (Toplam {total_weeks} Hafta)")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Referer": "https://arsiv.mackolik.com/"
    }

    for week in range(1, total_weeks + 1):
        # API'ye istek atıyoruz
        params = {
            "command": "getWeeklyStanding",
            "id": league_id,        # Lig ID (Süper lig için genelde 1)
            "seasonId": season_id,  # Sezon ID (URL'de s=70381 yazan yer)
            "week": week
        }

        try:
            response = requests.get(base_url, params=params, headers=headers)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')

                # Tablodaki maç skor linklerini bul (skor varsa maç oynanmıştır)
                # Genelde linkler 'matchScore' class'ına sahiptir veya skora link verilir.
                links = soup.find_all("a", href=True)

                week_count = 0
                for link in links:
                    href = link['href']
                    # Maç linki formatı kontrolü (/Mac/ veya /mac/)
                    if "/Mac/" in href or "/mac/" in href:
                        full_link = f"https://arsiv.mackolik.com{href}"
                        if full_link not in all_match_links:
                            all_match_links.append(full_link)
                            week_count += 1

                print(f"✅ {week}. Hafta tarandı: {week_count} maç bulundu.")
            else:
                print(f"⚠️ {week}. Hafta çekilemedi. Hata kodu: {response.status_code}")

        except Exception as e:
            print(f"❌ Hata ({week}. Hafta): {e}")

        # Sunucuyu yormamak için minik bir bekleme
        time.sleep(0.5)

    print(f"🎉 TOPLAM BULUNAN MAÇ SAYISI: {len(all_match_links)}")
    return all_match_links

# --- KULLANIM ---
# ÖRNEK: Türkiye Süper Lig 2025/2026 (Attığın HTML'den aldığım ID'ler)
URL: .../Puan-Durumu/s=70381/Turkiye-Trendyol-Super-Lig
LIG_ID = 1        # Türkiye Ligi ID'si genelde 1'dir.
SEZON_ID = 70381  # URL'deki s=... kısmı
TOPLAM_HAFTA = 38 # Ligin hafta sayısı

match_list = get_season_links(LIG_ID, SEZON_ID, TOPLAM_HAFTA)

# İlk 5 linki görelim
print("\nÖrnek Linkler:")
for l in match_list[:5]:
    print(l)